# DRC — Evaluation & analysis (curves, clusters, figures)

Turns trained models into results. Stages: **eval, ngram, hill, model_comparison, clustering, decision, figures**. Note `ngram` only needs the corpora, so it runs even if training never finished. `eval` and the analysis stages need the model manifest, so attach `kaggle_01_train`'s output (which also carries the data) before running.

**This notebook is resumable.** Every stage runs in its own subprocess via
`drc.pipeline.run_pipeline`, so a CUDA OOM or a hard kernel crash in one stage
takes down that subprocess and *nothing else* — the kernel survives, the
failure is recorded, and the rest of the pipeline proceeds where it can. If
Kaggle's 12-hour session limit cuts you off, just **re-run the notebook**:
finished stages detect their own outputs and skip, so you pick up where you
stopped instead of redoing hours of work.

**Scope:** `only=["eval", "ngram", "hill", "model_comparison", "clustering", "decision", "figures"]`.

## Before you run

- Set the accelerator to **GPU T4 x2** (Settings -> Accelerator -> *GPU T4 x2*).
  With two cards the training sweep runs two jobs in parallel; with one it falls
  back to a 48-run single-GPU plan.
- **T4 is a Turing GPU and does not support bf16.** The shipped
  `configs/base.yaml` sets `precision: bf16`. Before training on T4, change that
  line to `precision: fp16`. The GPU-detect cell below reminds you; it does not
  edit the config for you.

## Chaining notebooks on Kaggle

The pipeline is split across notebooks so a long run survives the session limit
and each phase can be re-run on its own. They chain through Kaggle's
**notebook-output datasets**:

1. Run `kaggle_00_data` to completion. Its `/kaggle/working` is saved as a
   notebook-output dataset when the run finishes (*Save Version*).
2. In the next notebook (`kaggle_01_train`), add that output dataset as an input
   (*Add Input -> Your Datasets*). It lands under `/kaggle/input/<dataset>/`.
3. The **restore-prior-artifacts** cell copies any `data/`, `models/`, and
   `results/` it finds under `/kaggle/input/*/` into the working repo, so the
   already-done stages resume as *skipped* and this notebook only does its part.
4. Repeat: `kaggle_01_train`'s output feeds `kaggle_02_eval_analysis`.

Nothing here fabricates results. A blocked or failed stage produces no numbers —
it just says so in the dashboard and lets the rest proceed.

In [ ]:
# --- Setup: make the `drc` package importable, idempotently. -------------------
# Safe to re-run. If `drc` already imports we do nothing heavy. Otherwise we find
# the repo (uploaded as a dataset, or already cloned) or clone it, then install.
import os, sys, glob, subprocess
from pathlib import Path

# Replace this with your repo's clone URL (used only if the repo isn't already
# present as a Kaggle dataset or a prior checkout under /kaggle/working).
REPO_URL = "https://github.com/your-org/drc-emnlp-2026.git"  # <-- EDIT ME
WORK = Path("/kaggle/working/drc-emnlp-2026")

# Eval needs torch; analysis needs scipy/sklearn/matplotlib for fits + figures.
EXTRA = "train"               # package extra to try, or "" for none
PIP_PACKAGES = "scipy scikit-learn matplotlib"  # plain pip fallbacks this notebook needs


def _have_drc() -> bool:
    try:
        import drc  # noqa: F401
        return True
    except Exception:
        return False


def _find_repo() -> Path | None:
    """Look for an existing checkout: a uploaded dataset or a prior /kaggle/working."""
    candidates = []
    # A repo uploaded as a Kaggle dataset shows up under /kaggle/input/<name>/...
    candidates += glob.glob("/kaggle/input/*/drc-emnlp-2026")
    candidates += glob.glob("/kaggle/input/*/src/drc")   # repo root contains src/drc
    candidates += [str(WORK)]
    for c in candidates:
        c = Path(c)
        # Normalise: we want the repo ROOT (the dir that contains src/drc).
        root = c
        if root.name == "drc" and root.parent.name == "src":
            root = root.parent.parent
        if (root / "src" / "drc").exists() or (root / "pyproject.toml").exists():
            return root
    return None


def _run(cmd: str) -> int:
    print("$", cmd)
    return subprocess.call(cmd, shell=True)


if _have_drc():
    print("drc already importable — skipping repo setup.")
    # Still try to locate WORK so os.chdir below lands somewhere sensible.
    found = _find_repo()
    if found is not None:
        WORK = found
else:
    repo = _find_repo()
    if repo is not None and repo != WORK:
        print(f"Found repo at {repo} (not cloning).")
        WORK = repo
    elif repo is None:
        print(f"No local repo found; cloning {REPO_URL} -> {WORK}")
        WORK.parent.mkdir(parents=True, exist_ok=True)
        _run(f'git clone --depth 1 "{REPO_URL}" "{WORK}"')
    else:
        print(f"Using existing checkout at {WORK}")

    # Try an editable install with the extra; degrade gracefully on any failure.
    installed = False
    if EXTRA:
        rc = subprocess.call(
            f'pip install -q -e "{WORK}"[{EXTRA}]', shell=True
        )
        installed = rc == 0
        if not installed:
            print(f"[warn] editable install with [{EXTRA}] failed; trying plain -e")
    if not installed:
        rc = subprocess.call(f'pip install -q -e "{WORK}"', shell=True)
        installed = rc == 0
    if not installed:
        # Last resort: don't install, just put src/ on the path so imports work.
        src = str(WORK / "src")
        if src not in sys.path:
            sys.path.insert(0, src)
        print(f"[warn] pip install failed; added {src} to sys.path as fallback.")

    # Notebook-specific plain packages (e.g. stanza, scipy) on top of the base.
    if PIP_PACKAGES.strip():
        _run(f"pip install -q {PIP_PACKAGES}")

# Work from the repo root so the config's RELATIVE paths resolve under it.
os.chdir(WORK)
print("cwd:", os.getcwd())
print("drc importable:", _have_drc())

In [ ]:
# --- Restore prior artifacts so already-done stages resume as "skipped". -------
# When this notebook is chained after another, the previous notebook's
# /kaggle/working is attached as an input dataset under /kaggle/input/<name>/.
# We copy its data/ models/ results/ into our working repo. Safe to re-run;
# dirs_exist_ok lets it merge over an existing tree. Guarded so a missing input
# (e.g. when you run this notebook standalone) is a no-op, not an error.
import glob, shutil
from pathlib import Path

WORK = Path(os.getcwd())  # set by the setup cell
RESTORE_DIRS = ("data", "models", "results")

# Candidate source roots: a chained notebook-output dataset will contain the
# repo's working tree, either at the repo root or one level down.
sources = []
sources += glob.glob("/kaggle/input/*/drc-emnlp-2026")
sources += glob.glob("/kaggle/input/*")

restored = []
for src_root in sources:
    src_root = Path(src_root)
    if src_root.resolve() == WORK.resolve():
        continue
    for sub in RESTORE_DIRS:
        src = src_root / sub
        if src.is_dir():
            try:
                shutil.copytree(src, WORK / sub, dirs_exist_ok=True)
                restored.append(str(src))
            except Exception as exc:  # never let a restore failure stop the run
                print(f"[warn] could not restore {src}: {exc}")

if restored:
    print("Restored prior artifacts from:")
    for r in restored:
        print("  ", r)
else:
    print("No prior artifacts found to restore (fine if this is the first stage).")

In [ ]:
# --- Detect GPUs and pick the sweep mode. --------------------------------------
import subprocess

n_gpus = 0
try:
    out = subprocess.run(
        ["nvidia-smi", "-L"], capture_output=True, text=True, check=False
    )
    print(out.stdout.strip() or "(nvidia-smi returned no GPUs)")
    n_gpus = sum(1 for ln in out.stdout.splitlines() if ln.strip().startswith("GPU "))
except FileNotFoundError:
    print("nvidia-smi not found — assuming no GPU (CPU-only).")

SINGLE_GPU = n_gpus < 2
print(f"\nDetected {n_gpus} GPU(s). SINGLE_GPU = {SINGLE_GPU}")

if n_gpus >= 2:
    print("Dual-GPU: the sweep runs two training jobs in parallel, one per card.")
elif n_gpus == 1:
    print("Single-GPU: the sweep uses the 48-run fallback (drops dose=4).")
else:
    print("No GPU: training/eval stages will fail; set Accelerator to GPU T4 x2.")

print(
    "\n[CAVEAT] T4 is a Turing card and does NOT support bf16. The shipped"
    "\n         configs/base.yaml uses precision: bf16. Before training on T4,"
    "\n         edit that line to  precision: fp16  (this cell does not edit it)."
)

In [ ]:
# --- Run the pipeline. ---------------------------------------------------------
# default_phases() returns the 13 wired stages; run_pipeline() isolates failures,
# skips finished stages, blocks stages with unmet deps, and prints a dashboard.
# It never raises on a stage failure, so this cell completes even if a stage dies.
from pathlib import Path
from drc.pipeline import default_phases, run_pipeline

cfg = Path("configs/base.yaml")
status = Path("results/pipeline_status.json")

# Evaluation and analysis stages.
only = ["eval", "ngram", "hill", "model_comparison", "clustering", "decision", "figures"]

stages = default_phases(cfg, single_gpu=SINGLE_GPU)
results = run_pipeline(stages, status_path=status, only=only)
# The dashboard is already printed above by run_pipeline.

In [ ]:
# --- Inspect what we produced. -------------------------------------------------
# Tolerant of missing files: a fresh or partial run just shows fewer artifacts.
import json
from pathlib import Path

status_path = Path("results/pipeline_status.json")
if status_path.exists():
    data = json.loads(status_path.read_text())
    print("Pipeline status:")
    for name, r in data.items():
        secs = f"{r.get('seconds', 0):.1f}s" if r.get("seconds") else ""
        detail = f"  {r['detail']}" if r.get("detail") else ""
        print(f"  {r['status']:<8} {name:<18} {secs}{detail}")
else:
    print("No results/pipeline_status.json yet — has the run cell completed?")

for d in ("results", "results/figures"):
    p = Path(d)
    if p.is_dir():
        items = sorted(x.name for x in p.iterdir())
        print(f"\n{d}/ ({len(items)} items):")
        for it in items:
            print("  ", it)
    else:
        print(f"\n{d}/ does not exist yet.")

decision = Path("results/decision.txt")
if decision.exists():
    print("\n=== results/decision.txt ===")
    print(decision.read_text())